In [ ]:
import os
from dotenv import load_dotenv

# Carrega as variáveis do .env
load_dotenv()

# Caminho para o modelo salvo
training_model = os.getenv("TRAINING_MODEL")

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

# Carrega tokenizer e modelo treinado
tokenizer = BertTokenizer.from_pretrained(training_model)
model = BertForSequenceClassification.from_pretrained(training_model)

# Coloca o modelo em modo de inferência
model.eval()  

In [ ]:
import torch

def predict_sentiment(text):
    # Tokeniza a entrada
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    )

    # Inferência sem calcular gradiente
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0][predicted_class].item()

    # Ajuste conforme seu mapeamento de rótulo
    label = "Positivo" if predicted_class == 1 else "Negativo"
    return {
        "label": label,
        "confidence": round(confidence * 100, 2)
    }


In [ ]:
print(predict_sentiment("This movie was absolutely fantastic!"))
print(predict_sentiment("I hated the entire experience."))
print(predict_sentiment("It was okay, not great but not bad either."))